In [1]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#      https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Categorical Evaluation Demo

<table align="left">

  <td>
    <a href="https://colab.research.google.com/github/haiyuan-eng-google/BigQuery-Agent-Analytics-SDK/blob/main/examples/categorical_evaluation_demo.ipynb">
      <img src="https://raw.githubusercontent.com/googleapis/python-bigquery-dataframes/refs/heads/main/third_party/logo/colab-logo.png" alt="Colab logo"> Run in Colab
    </a>
  </td>
  <td>
    <a href="https://github.com/haiyuan-eng-google/BigQuery-Agent-Analytics-SDK/blob/main/examples/categorical_evaluation_demo.ipynb">
      <img src="https://raw.githubusercontent.com/googleapis/python-bigquery-dataframes/refs/heads/main/third_party/logo/github-logo.png" width="32" alt="GitHub logo">
      View on GitHub
    </a>
  </td>
  <td>
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/haiyuan-eng-google/BigQuery-Agent-Analytics-SDK/main/examples/categorical_evaluation_demo.ipynb">
      <img src="https://www.gstatic.com/images/branding/product/1x/google_cloud_48dp.png" alt="Vertex AI logo" width="32">
      Open in Vertex AI Workbench
    </a>
  </td>
  <td>
    <a href="https://console.cloud.google.com/bigquery/import?url=https://github.com/haiyuan-eng-google/BigQuery-Agent-Analytics-SDK/blob/main/examples/categorical_evaluation_demo.ipynb">
      <img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcTW1gvOovVlbZAIZylUtf5Iu8-693qS1w5NJw&s" alt="BQ logo" width="35">
      Open in BQ Studio
    </a>
  </td>
</table>

## Install Dependencies

In [2]:
!pip install -q bigquery-agent-analytics google-cloud-bigquery nest-asyncio

DEPRECATION: Configuring installation scheme with distutils config files is deprecated and will no longer work in the near future. If you are using a Homebrew or Linuxbrew Python, please see discussion at https://github.com/Homebrew/homebrew-core/issues/76621


ERROR: Could not find a version that satisfies the requirement bigquery-agent-analytics (from versions: none)



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: /usr/local/opt/python@3.9/bin/python3.9 -m pip install --upgrade pip
ERROR: No matching distribution found for bigquery-agent-analytics


## Authenticate & Configure

In [3]:
import os

# Colab authentication
try:
    from google.colab import auth
    auth.authenticate_user()
    print("Colab authentication successful.")
except ImportError:
    print("Not running in Colab — using default credentials.")

# ---------- Configuration ----------
PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "test-project-0728-467323")
DATASET_ID = os.environ.get("BQ_DATASET", "agent_analytics")
TABLE_ID = os.environ.get("BQ_TABLE", "agent_events")
MODEL_NAME = os.environ.get("MODEL_NAME", "gemini-3-flash-preview")
LOCATION = "US"

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "true"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"

# Enable async in Jupyter
import nest_asyncio
nest_asyncio.apply()

print(f"Project  : {PROJECT_ID}")
print(f"Dataset  : {DATASET_ID}")
print(f"Table    : {TABLE_ID}")
print(f"Model    : {MODEL_NAME}")

Not running in Colab — using default credentials.
Project  : test-project-0728-467323
Dataset  : agent_analytics
Table    : agent_events
Model    : gemini-3-flash-preview


## 1. Define Categorical Metrics

Categorical evaluation classifies agent sessions into user-defined categories
rather than assigning numeric scores. Each **`CategoricalMetricDefinition`**
declares a metric name, a human-readable definition, and a list of
**`CategoricalMetricCategory`** values that the classifier is allowed to return.

Under the hood, the SDK translates these definitions into:
- **`AI.CLASSIFY`** calls (when `include_justification=False`) — BigQuery's
  native classification operator, or
- **`AI.GENERATE`** calls (when justification is requested) — structured
  generation with strict category validation.

Below we define two metrics:
1. **response_quality** — rates overall agent response quality.
2. **task_completion** — classifies whether the agent completed the user's request.

In [4]:
from bigquery_agent_analytics import (
    CategoricalMetricCategory,
    CategoricalMetricDefinition,
)

# Metric 1: Response Quality
response_quality = CategoricalMetricDefinition(
    name="response_quality",
    definition="Overall quality of the agent's responses in the session.",
    categories=[
        CategoricalMetricCategory(
            name="excellent",
            definition="Responses are accurate, comprehensive, and well-structured.",
        ),
        CategoricalMetricCategory(
            name="good",
            definition="Responses are mostly accurate with minor gaps or formatting issues.",
        ),
        CategoricalMetricCategory(
            name="poor",
            definition="Responses are inaccurate, incomplete, or unhelpful.",
        ),
    ],
)

# Metric 2: Task Completion
task_completion = CategoricalMetricDefinition(
    name="task_completion",
    definition="Whether the agent fulfilled the user's request.",
    categories=[
        CategoricalMetricCategory(
            name="complete",
            definition="The agent fully addressed all parts of the user's request.",
        ),
        CategoricalMetricCategory(
            name="partial",
            definition="The agent addressed some parts but missed others.",
        ),
        CategoricalMetricCategory(
            name="failed",
            definition="The agent did not meaningfully address the user's request.",
        ),
    ],
)

metrics = [response_quality, task_completion]

print(f"Defined {len(metrics)} categorical metrics:")
for m in metrics:
    cats = [c.name for c in m.categories]
    print(f"\n  [{m.name}]  categories: {cats}")
    print(f"    Definition : {m.definition}")
    print(f"    Required   : {m.required}")
    for c in m.categories:
        print(f"      - {c.name}: {c.definition}")

Defined 2 categorical metrics:

  [response_quality]  categories: ['excellent', 'good', 'poor']
    Definition : Overall quality of the agent's responses in the session.
    Required   : True
      - excellent: Responses are accurate, comprehensive, and well-structured.
      - good: Responses are mostly accurate with minor gaps or formatting issues.
      - poor: Responses are inaccurate, incomplete, or unhelpful.

  [task_completion]  categories: ['complete', 'partial', 'failed']
    Definition : Whether the agent fulfilled the user's request.
    Required   : True
      - complete: The agent fully addressed all parts of the user's request.
      - partial: The agent addressed some parts but missed others.
      - failed: The agent did not meaningfully address the user's request.


## 2. Configure & Run Evaluation

**`CategoricalEvaluationConfig`** bundles the metric definitions with
execution parameters (model endpoint, temperature, persistence settings).

The SDK client's **`evaluate_categorical()`** method executes a tiered fallback:

| Justification | Execution cascade |
|---|---|
| `include_justification=False` | `AI.CLASSIFY` -> `AI.GENERATE` -> Gemini API |
| `include_justification=True` (default) | `AI.GENERATE` -> Gemini API |

- **`AI.CLASSIFY`** — BigQuery's native classification operator. Returns exact
  category labels with no JSON parsing needed. Does not support justification.
- **`AI.GENERATE`** — Structured generation with a classification prompt.
  Supports justification and validates returned categories against the allowed set.
- **Gemini API** — Direct API fallback when BigQuery-native execution is
  unavailable.

In [5]:
import asyncio

from bigquery_agent_analytics import (
    Client,
    CategoricalEvaluationConfig,
    TraceFilter,
)

# Initialise the SDK client
client = Client(
    project_id=PROJECT_ID,
    dataset_id=DATASET_ID,
    table_id=TABLE_ID,
    location=LOCATION,
    endpoint=MODEL_NAME,
)
print("SDK Client initialised.")

# Build the categorical evaluation config
cat_config = CategoricalEvaluationConfig(
    metrics=metrics,
    endpoint=MODEL_NAME,
    temperature=0.0,
    include_justification=True,   # Use AI.GENERATE path for justifications
    persist_results=False,         # Set True to write results to BigQuery
)

print(f"\nCategoricalEvaluationConfig:")
print(f"  Metrics              : {[m.name for m in cat_config.metrics]}")
print(f"  Endpoint             : {cat_config.endpoint}")
print(f"  Temperature          : {cat_config.temperature}")
print(f"  Include justification: {cat_config.include_justification}")
print(f"  Persist results      : {cat_config.persist_results}")
print(f"  Prompt version       : {cat_config.prompt_version}")

# Run categorical evaluation
# Optionally pass a TraceFilter with session_ids to scope the evaluation.
try:
    report = client.evaluate_categorical(
        config=cat_config,
        filters=TraceFilter(),  # Evaluates all available sessions
    )
    print(f"\nEvaluation complete!")
    print(f"  Execution mode : {report.details.get('execution_mode', 'unknown')}")
    print(f"  Total sessions : {report.total_sessions}")
except Exception as exc:
    print(f"\nCategorical evaluation failed: {exc}")
    print("This is expected if no agent traces exist in the table yet.")
    report = None

SDK Client initialised.

CategoricalEvaluationConfig:
  Metrics              : ['response_quality', 'task_completion']
  Endpoint             : gemini-3-flash-preview
  Temperature          : 0.0
  Include justification: True
  Persist results      : False
  Prompt version       : None



Evaluation complete!
  Execution mode : api_fallback
  Total sessions : 18


## 3. Analyze Results

The evaluation returns a **`CategoricalEvaluationReport`** containing:

- **`total_sessions`** — number of sessions evaluated.
- **`category_distributions`** — maps each metric name to `{category: count}`.
- **`session_results`** — list of **`CategoricalSessionResult`**, each holding
  per-metric **`CategoricalMetricResult`** objects.
- **`details`** — execution metadata (mode, parse errors, fallback reasons).
- **`summary()`** — human-readable text summary.

Each `CategoricalMetricResult` includes:

| Field | Description |
|---|---|
| `metric_name` | Which metric was evaluated |
| `category` | Assigned category label (or `None` on failure) |
| `passed_validation` | Whether the category matched an allowed value |
| `justification` | Model's explanation (when `include_justification=True`) |
| `parse_error` | Whether the model's response could not be parsed |
| `raw_response` | The raw string returned by the model |

In [6]:
if report is not None:
    # Print the human-readable summary
    print("=" * 60)
    print(report.summary())
    print("=" * 60)

    # Inspect category distributions
    print("\nCategory Distributions:")
    for metric_name, dist in report.category_distributions.items():
        print(f"\n  [{metric_name}]")
        total = sum(dist.values())
        for category, count in sorted(dist.items(), key=lambda x: -x[1]):
            pct = (count / total * 100) if total > 0 else 0.0
            print(f"    {category:15s}: {count:3d}  ({pct:.1f}%)")

    # Inspect per-session results (CategoricalSessionResult)
    print(f"\nPer-Session Results ({len(report.session_results)} sessions):")
    for sr in report.session_results[:5]:  # Show first 5 sessions
        print(f"\n  Session: {sr.session_id}")
        for mr in sr.metrics:
            status = "PASS" if mr.passed_validation else "FAIL"
            print(f"    {mr.metric_name}: {mr.category} [{status}]")
            if mr.justification:
                print(f"      Justification: {mr.justification[:200]}")
            if mr.parse_error:
                print(f"      Parse error detected")

    if len(report.session_results) > 5:
        print(f"\n  ... and {len(report.session_results) - 5} more sessions.")

    # Execution details
    print("\nExecution Details:")
    for key, value in report.details.items():
        print(f"  {key}: {value}")

    print(f"\nReport created at: {report.created_at.isoformat()}")
else:
    print("No report available — run the evaluation cell first.")

Categorical Evaluation Report: categorical_evaluator
  Dataset: test-project-0728-467323.agent_analytics.agent_events WHERE TRUE
  Sessions: 18
  Parse errors: 2 (5.6%)
  Category Distributions:
    response_quality:
      excellent: 17
    task_completion:
      complete: 17

Category Distributions:

  [response_quality]
    excellent      :  17  (100.0%)

  [task_completion]
    complete       :  17  (100.0%)

Per-Session Results (18 sessions):

  Session: adcp-f5a6b8bd92e8
    response_quality: excellent [PASS]
      Justification: The agent provided a highly professional, well-structured response. It included a detailed breakdown of inventory, audience matching, and a clear budget allocation table with specific metrics like CPM
    task_completion: complete [PASS]
      Justification: The agent fulfilled all requirements of the user's brief: it checked inventory for the three specific products, performed audience matching, and provided a recommended budget split across all three.



## 4. Dashboard Views

The **`CategoricalViewManager`** creates standard BigQuery views over the
categorical results table for dashboard integration. Views include:

| View | Description |
|---|---|
| `categorical_results_latest` | Dedup base view — keeps only the latest result per (session, metric, prompt_version) |
| `categorical_daily_counts` | Daily category counts per metric and execution mode |
| `categorical_hourly_counts` | Hourly category counts per metric |
| `categorical_operational_metrics` | Parse errors, validation failures, and fallback rates by day |

All downstream views read from the dedup base view
(`categorical_results_latest`), so retries and overlapping micro-batch runs
do not inflate counts.

In [7]:
from bigquery_agent_analytics import CategoricalViewManager

# Initialise the view manager
vm = CategoricalViewManager(
    project_id=PROJECT_ID,
    dataset_id=DATASET_ID,
    results_table="categorical_results",  # Default results table name
    view_prefix="",                        # Optional prefix, e.g. "adk_"
    location=LOCATION,
)

# List available views
print("Available dashboard views:")
for view_name in vm.available_views():
    print(f"  - {view_name}")

# Preview the SQL for each view
print("\n" + "=" * 60)
for view_name in vm.available_views():
    print(f"\n--- {view_name} ---")
    try:
        sql = vm.get_view_sql(view_name)
        print(sql)
    except Exception as exc:
        print(f"  Error: {exc}")

# Create all views in BigQuery (uncomment to execute)
# try:
#     created = vm.create_all_views()
#     print(f"\nCreated {len(created)} views:")
#     for base_name, full_name in created.items():
#         print(f"  {base_name} -> {PROJECT_ID}.{DATASET_ID}.{full_name}")
# except Exception as exc:
#     print(f"View creation failed: {exc}")

print("\nTo create views in BigQuery, uncomment the create_all_views() block above.")

Available dashboard views:
  - categorical_results_latest
  - categorical_daily_counts
  - categorical_hourly_counts
  - categorical_operational_metrics


--- categorical_results_latest ---
CREATE OR REPLACE VIEW `test-project-0728-467323.agent_analytics.categorical_results_latest` AS
WITH ranked AS (
  SELECT *,
    ROW_NUMBER() OVER (
      PARTITION BY session_id, metric_name, COALESCE(prompt_version, '')
      ORDER BY created_at DESC, raw_response DESC
    ) AS _rn
  FROM `test-project-0728-467323.agent_analytics.categorical_results`
)
SELECT * EXCEPT(_rn) FROM ranked WHERE _rn = 1


--- categorical_daily_counts ---
CREATE OR REPLACE VIEW `test-project-0728-467323.agent_analytics.categorical_daily_counts` AS
SELECT
  DATE(created_at) AS eval_date,
  metric_name,
  category,
  execution_mode,
  COUNT(*) AS session_count
FROM `test-project-0728-467323.agent_analytics.categorical_results_latest`
GROUP BY 1, 2, 3, 4


--- categorical_hourly_counts ---
CREATE OR REPLACE VIEW `test-proj

## Summary

This notebook demonstrated **Section 17: Categorical Evaluation** of the
BigQuery Agent Analytics SDK.

| Step | Component | Description |
|---|---|---|
| 1 | **`CategoricalMetricDefinition`** | Define metrics with named categories and descriptions |
| 2 | **`CategoricalEvaluationConfig`** | Bundle metrics with endpoint, temperature, and persistence settings |
| 3 | **`Client.evaluate_categorical()`** | Run evaluation via AI.CLASSIFY / AI.GENERATE / Gemini API fallback |
| 4 | **`CategoricalEvaluationReport`** | Inspect distributions, per-session results, and execution details |
| 5 | **`CategoricalViewManager`** | Create dedup + aggregation BigQuery views for dashboards |

### Key Takeaways

- **Label-valued results**: Unlike numeric evaluators, categorical evaluation returns
  discrete labels with strict validation against allowed categories.
- **3-tier execution cascade**: The SDK automatically falls back from `AI.CLASSIFY`
  to `AI.GENERATE` to Gemini API, ensuring evaluation works across environments.
- **Justification support**: Set `include_justification=True` to get model explanations
  for each classification (uses the `AI.GENERATE` path).
- **Dashboard-ready views**: `CategoricalViewManager` creates deduplication and
  aggregation views that connect directly to Looker Studio or other BI tools.
- **Persistence**: Set `persist_results=True` and provide a `results_table` to write
  classification results to BigQuery for longitudinal tracking.

In [8]:
print("Notebook complete!")

Notebook complete!
